# 📊 5장.클로드 데스크톱으로 나만의 투자 MCP를 만들어 실시간 주식 데이터 분석하기
> **핵심 학습 목표**
> 1. **도구 환경 구성**: Node.js(`npx`)와 Astral `uv`(`uvx`) 설치 및 검증
> 2. **외부 데이터 MCP 연동**: 최신 웹 검색 및 추출(`Firecrawl`), 금융 시계열 데이터(`Alpha Vantage`)
> 3. **AI 퀀트 투자 분석**: MCP 연동 환경에서 실전 멀티팩터 모델 및 공급망 분석 프롬프트 실습
> 4. **커스텀 MCP 서버 개발**: 네이버 금융 기반의 `korea-stock-mcp` 제작, 로컬 테스트 및 배포 연동

---

## 🛠️ 1. 도구 설치 및 환경 구성

Firecrawl, Alpha Vantage 등 외부 데이터 소스와 연동하기 위해 **Node.js**(`npx`) 및 초고속 패키지 매니저 **uv**(`uvx`) 환경을 구성합니다.


### 1-1. Node.js & npm 설치 확인 (PowerShell)

`firecrawl-mcp` 등 Node.js 기반 MCP 패키지를 실행하기 위해 버전이 정상적으로 출력되는지 확인합니다.

```powershell
# Node.js 및 npm 버전 확인
node -v
npm -v
```


### 1-2. Astral uv 설치 (Python MCP 패키지 실행기)

Alpha Vantage MCP(`av-mcp`)와 같은 파이썬 기반 MCP 도구를 가상환경 구축 없이 즉시 실행(`uvx`)하기 위해 Astral `uv`를 설치합니다.

```powershell
# PowerShell에서 Astral uv 설치 스크립트 실행
iwr https://astral.sh/uv/install.ps1 -UseBasicParsing | iex
```


### 1-3. uv 버전 확인

설치가 완료된 후 `uv` 명령어가 정상 동작하는지 확인합니다.

```powershell
uv --version
```


---

## ⚙️ 2. Claude 데스크톱 MCP 연동 설정

Claude 데스크톱 앱의 설정 파일에 사용할 MCP 서버들을 등록합니다.

> 📁 **설정 파일 경로 (`claude_desktop_config.json`)**
> - **Windows**: `%APPDATA%\Claude\claude_desktop_config.json`
> - **macOS**: `~/Library/Application Support/Claude/claude_desktop_config.json`

### 2-1. Firecrawl MCP 등록 (웹 크롤링 & 실시간 데이터 추출)

웹 페이지의 최신 금융 뉴스와 리서치 자료를 크롤링하여 LLM에 제공하는 `firecrawl-mcp` 설정입니다.

```json
{
  "mcpServers": {
    "firecrawl-mcp": {
      "command": "npx",
      "args": ["-y", "firecrawl-mcp"],
      "env": {
        "FIRECRAWL_API_KEY": "{내 firecrawl API KEY}"
      }
    }
  }
}
```


### 2-2. Alpha Vantage MCP 추가 등록 (주가 및 재무 시계열 데이터)

Firecrawl과 함께 주가 및 재무 지표를 실시간 조회할 수 있는 `alphavantage` MCP를 추가한 최종 설정입니다.

```json
{
  "mcpServers": {
    "firecrawl-mcp": {
      "command": "npx",
      "args": ["-y", "firecrawl-mcp"],
      "env": {
        "FIRECRAWL_API_KEY": "{내 firecrawl API KEY}"
      }
    },
    "alphavantage": {
      "command": "uvx",
      "args": ["av-mcp", "{내 alpha vantage API KEY}"]
    }
  }
}
```


---

## 📈 3. 실전 AI 퀀트 투자 분석 프롬프트

등록한 **Firecrawl**(뉴스·이슈 수집)과 **Alpha Vantage**(주가·재무 수집)를 활용하여 심도 있는 투자 분석을 수행하는 프롬프트 예시입니다.

### 3-1. 미국 경제 & 멀티팩터 퀀트 투자 중요도 분석 프롬프트

| 팩터 구분 | 대상 세부 지표 | 활용 목적 |
| :--- | :--- | :--- |
| **모멘텀 (Momentum)** | 주가 추세, 거래량 | 시장의 단기 수급 및 추세 강도 측정 |
| **밸류 (Value)** | PER, PBR | 기업 펀더멘털 대비 주가 저평가 수준 평가 |
| **퀄러티 (Quality)** | 매출액 증가율, 영업이익 증가율, 순이익 증가율 | 기업의 수익성 및 지속 가능한 성장성 검증 |

```markdown
# 지시문
- 당신은 미국 경제 전문가이자 퀀트 투자 전략가입니다.
- 최근 미국 주식시장의 뉴스와 관련 자료를 학습하여, 멀티팩터 세부 지표별로 주가 예측 중요도를 분석해주세요.
- 분석 결과는 멀티팩터 세부 지표의 가중치 조정에 활용할 예정입니다.

# 제약 조건
- 최근 3개월간의 미국 경제, 금융, 주식시장의 주요 이슈와 뉴스를 수집해서 분석합니다.
- 오늘은 [2025년12월 16일] 입니다. Firecrawl MCP 를 활용하여 뉴스 자료를 수집합니다.   
- Alpha Vantage MCP를 활용하여 주가정보를 수집합니다.
- 멀티팩터 지표는 다음과 같이 7가지로 정의 합니다.
  - 모멘텀 지표 : 주가, 거래량
  - 밸류 지표 : PER, PBR
  - 퀄러티 지표 : 매출액 증가율, 영업이익 증가율, 순이익 증가율
- 앞으로 3개월간의 주가를 예측할 때, 멀티팩터 세부 지표별 중요도를 계량적으로 분석합니다.
- 보고서는 객관적인 데이터와 발표자료를 근거로 출력해주세요.

# 출력형식
- 멀티팩터 지표를 나열하고, 각 지표별로 주가 예측에 대한 중요도를 퍼센트 비중으로 표현합니다.
- 멀티팩터 지표를 나열하고, 중요도를 선정한 근거를 제시합니다.
```


### 3-2. 반도체 업종 공급망(엔비디아 & AMD) 수혜주 발굴 프롬프트

```markdown
# 지시문
- 당신은 증권사에서 반도체 업종을 담당하고 있는 리서치 연구원입니다.
- 반도체 업종에서 엔비디아와 AMD 의 공급망 분석에 기반하여 투자유망 종목을 추천해주세요.

# 제약 조건
- 엔비디아와 AMD를 포함하여 반도체 업종의 기업과 관련된 뉴스, 리서치보고서, 재무데이터를 수집해서 분석합니다.  
- Firecrawl 과 Alpha Vantage를 통하여 반드시 최신 정보를 찾아야 합니다.
- 오늘은 [2025.12.18] 입니다.
- 보고서는 객관적인 데이터와 발표자료를 근거로 출력해주세요.

# 출력형식
- 반도체 업종에서 엔비디아와 AMD 의 공급 사슬 내 기업 중에서, 가장 큰 수혜가 예상되는 주식을 3개 선정합니다.
- 선정된 주식별로 기업개요, 주요 이슈를 정리합니다.
- 선정된 기업별로 현재 주가와 향후 목표주가를 제안합니다.
- 최종적으로 투자전망 측면에서 1개 기업을 선정하고, 선정 사유와 유의사항을 출력합니다.
```


### 3-3. Alpha Vantage 응답 데이터 예시 (TSMC / Symbol: TSM)

Alpha Vantage MCP를 통해 수집된 TSMC의 기업 개요(Company Overview), 밸류에이션(PER, PBR), 재무 건전성 및 컨센서스 데이터 예시입니다.

```json
{
    "Symbol": "TSM",
    "AssetType": "Common Stock",
    "Name": "Taiwan Semiconductor Manufacturing",
    "Description": "Taiwan Semiconductor Manufacturing Company, Limited (TSMC) stands as the foremost pure-play semiconductor foundry globally, headquartered in Hsinchu, Taiwan. Renowned for its cutting-edge manufacturing and design expertise, TSMC serves a wide array of industries, including telecommunications, high-performance computing, and automotive, playing a crucial role in the global technology ecosystem. The company's relentless focus on innovation is evident through its significant investments in advanced technologies and processes, ensuring its leadership in semiconductor fabrication. With a robust international presence and a diverse investor base, TSMC continues to forge strategic partnerships that enhance its influence and adaptability within the evolving semiconductor landscape.",
    "CIK": "1046179",
    "Exchange": "NYSE",
    "Currency": "USD",
    "Country": "USA",
    "Sector": "TECHNOLOGY",
    "Industry": "SEMICONDUCTORS",
    "Address": "HSINCHU SCIENCE PARK, HSINCHU CITY, TAIWAN, 300096",
    "OfficialSite": "https://www.tsmc.com",
    "FiscalYearEnd": "December",
    "LatestQuarter": "2025-09-30",
    "MarketCapitalization": "1487852536000",
    "EBITDA": "2484714078000",
    "PERatio": "29.67",
    "PEGRatio": "1.403",
    "BookValue": "6.15",
    "DividendPerShare": "20.5",
    "DividendYield": "0.0117",
    "EPS": "9.67",
    "RevenuePerShareTTM": "87.54",
    "ProfitMargin": "0.433",
    "OperatingMarginTTM": "0.506",
    "ReturnOnAssetsTTM": "0.166",
    "ReturnOnEquityTTM": "0.347",
    "RevenueTTM": "3631424995000",
    "GrossProfitTTM": "2141685285000",
    "DilutedEPSTTM": "9.67",
    "QuarterlyEarningsGrowthYOY": "0.391",
    "QuarterlyRevenueGrowthYOY": "0.303",
    "AnalystTargetPrice": "344.57",
    "AnalystRatingStrongBuy": "5",
    "AnalystRatingBuy": "10",
    "AnalystRatingHold": "2",
    "AnalystRatingSell": "0",
    "AnalystRatingStrongSell": "0",
    "TrailingPE": "29.67",
    "ForwardPE": "23.87",
    "PriceToSalesRatioTTM": "0.41",
    "PriceToBookRatio": "9.32",
    "EVToRevenue": "9.84",
    "EVToEBITDA": "13.72",
    "Beta": "1.267",
    "52WeekHigh": "313.13",
    "52WeekLow": "132.98",
    "50DayMovingAverage": "293.52",
    "200DayMovingAverage": "231.74",
    "SharesOutstanding": "5186505000",
    "SharesFloat": "24172484000",
    "PercentInsiders": "0.039",
    "PercentInstitutions": "16.268",
    "DividendDate": "2026-04-09",
    "ExDividendDate": "2026-03-17"
}
```


---

## 🏗️ 4. 나만의 한국 주식 MCP 서버 개발 (`korea-stock-mcp`)

네이버 금융 데이터를 직접 스크래핑하여 Claude 등 LLM이 활용할 수 있는 독자적인 MCP 서버 패키지를 구축합니다.

### 4-1. 프로젝트 패키지 구조

```text
korea-stock-mcp/
├── package.json        # 패키지 메타데이터, 의존성 및 bin 실행 스크립트 정의
├── index.js            # MCP 서버 진입점 (핵심 로직: 데이터 수집 함수 구현 및 MCP 도구 노출)
└── README.md           # 서버 설명, 설치 가이드 및 사용법
```


### 4-2. MCP 서버 개발 요청 프롬프트 (네이버 금융 연동)

Claude Code나 ChatGPT 등에 전달하여 `korea-stock-mcp` 소스 코드를 생성하도록 요청하는 프롬프트입니다.

```markdown
네이버 금융에서 ETF, 테마, 시가총액 상위 종목 데이터 등을 추출하는 MCP 서버를 만들고자합니다.
MCP서버의 이름은 korea-stock-mcp 입니다.

1. Node.js 기반으로 index.js를 작성하고, package.json과 README.md까지 포함한 최소 패키지 구조를 생성해주세요.
2. index.js의 기능함수는 아래와 같습니다.
  - https://finance.naver.com/sise/etf.naver 에서 시가총액 순으로 ETF를 가져온다
  - https://finance.naver.com/sise/theme.naver 에서 전일대비 상승이 높은순으로 테마를 가져오고, 해당 테마의 주도주 종목명을 가져온다
  - https://finance.naver.com/sise/sise_market_sum.naver 에서 시가총액순으로 종목 이름과 현재가 정보를 가져온다
  - https://finance.naver.com/sise/dividend_list.naver 에서 배당수익률이 높은 순으로 종목정보를 가져온다
  - https://finance.naver.com/search/search.naver?query={종목명} 을 통해서 해당 종목에 대한 종목코드 정보를 가져온다
  - https://finance.naver.com/item/main.naver?code={종목코드} 방식을 통하여 해당 종목코드의 현재가 정보를 가져온다
3. 서버는 npx로 실행 가능해야 하며, MCP 프로토콜과 Claude가 함수를 이해할 수 있도록 메타데이터를 노출해주세요.

+ 네이버 금융은 UTF-8이 아니라 EUC-KR(또는 CP949) 로 되어 있음을 참고해
```


### 4-3. 패키지 설치 및 로컬 테스트 / 전역 링크

개발한 MCP 서버 폴더(`korea-stock-mcp/`) 내에서 아래 명령어들을 차례대로 실행합니다.

#### ① 의존성 패키지 설치
```bash
npm install
```

#### ② 서버 로컬 단독 실행 테스트
```bash
npm start
```

#### ③ 로컬 시스템 전역 심볼릭 링크 생성
로컬 개발 중인 패키지를 `npx korea-stock-mcp` 형태로 어디서든 즉시 실행할 수 있도록 전역 링크합니다.
```bash
npm link
```


---

## 🚀 5. Claude 데스크톱 연동 및 실전 질의

제작한 `korea-stock-mcp`를 Claude 데스크톱 설정 파일(`claude_desktop_config.json`)에 등록합니다.

### 5-1. 로컬 개발 MCP 서버 등록 (npm link 연동)

로컬에서 `npm link`로 연결된 `korea-stock-mcp`를 실행하도록 등록합니다.

```json
{
  "mcpServers": {
    "korea-stock-mcp": {
      "command": "npx",
      "args": ["korea-stock-mcp"],
      "env": {
        "NODE_NO_WARNINGS": "1"
      }
    }
  }
}
```


### 5-2. npm 배포 버전 MCP 서버 등록 (공식 배포 패키지)

npm 저장소에 배포된 최신 패키지(`@drfirst/korea-stock-mcp@latest`)를 사용하는 경우의 설정입니다.

```json
{
  "mcpServers": {
    // 기존 데이터가 있다면: 아래 블록만 추가
    "korea-stock-mcp": {
      "command": "npx",
      "args": ["-y", "@drfirst/korea-stock-mcp@latest"]
    }
    // 기존 데이터가 있다면: 여기까지!
  }
}
```


### 5-3. MCP 연동 실전 질의 프롬프트

설정이 완료된 후 Claude와 대화창에서 `korea-stock-mcp`의 네이버 금융 도구를 직접 호출하도록 질의합니다.

> 💬 **실전 질의 프롬프트**
>
> ```text
> korea-stock-mcp를 활용해서 오늘 주목받은 테마가 뭔지 알아봐
> ```
